# Getting LastFM Data
This notebook will ingest scrobbling data for a given username's LastFM account. 

## Setup
Load the API key and username from `lastfm.env`.

In [21]:
import os
import json
import time
import requests
import pandas as pd
from dotenv import load_dotenv
load_dotenv('lastfm.env')
API_KEY = os.environ['LASTFM_API_KEY']
USER = os.environ['LASTFM_USER']; print(f'User: {USER}')

URL = 'https://ws.audioscrobbler.com/2.0/'

User: sk3141


## Request parameters
`limit` is the number of scrobbles per page (max 200).

In [22]:
LIMIT = 200

params = {
    'method': 'user.getrecenttracks',
    'user': USER,
    'api_key': API_KEY,
    'format': 'json',
    'limit': LIMIT,
}

## Explore one response
The tracks are under `track`, and the pagination info is under `@attr`.

In [23]:
response = requests.get(URL, params=params, timeout=30)
print(response.status_code)

data = response.json()
#print(json.dumps(data, indent=2)[:3000])

200


In [24]:
print(data['recenttracks']['@attr'])
track = data['recenttracks']['track'][0]
print(track.keys())
track['artist']['#text']

{'perPage': '200', 'totalPages': '169', 'page': '1', 'user': 'sk3141', 'total': '33722'}
dict_keys(['artist', 'streamable', 'image', 'mbid', 'album', 'name', 'url', 'date'])


'Radiohead'

## Parse a track
Flatten one scrobble into a dict. The track that is currently playing has no `date`, so it is skipped.

In [25]:
def parse_track(track):
    '''Flatten one scrobble from the API into a simple dict.
    Returns None for the now-playing track, which has no timestamp yet.
    '''
    if 'date' not in track:  # now-playing tracks have no 'date' key
        return None
    return {
        'artist': track['artist']['#text'],
        'album': track['album']['#text'],
        'track': track['name'],
        'uts': int(track['date']['uts']),  # Unix seconds (UTC), as an int
    }

In [26]:
tracks = data['recenttracks']['track']
print(parse_track(tracks[0]))

{'artist': 'Radiohead', 'album': 'In Rainbows', 'track': 'Jigsaw Falling Into Place', 'uts': 1794199036}


## Fetch one page
`fetch_page` returns the parsed tracks for a single page. `raise_for_status()` raises an error if a request fails.

In [7]:
def fetch_page(page):
    '''Fetch one page of scrobbles and return them as a list of parsed dicts.'''
    page_params = {**params, 'page': page}  # copy so the shared params dict isn't mutated
    response = requests.get(URL, params=page_params, timeout=30)
    response.raise_for_status()  # raise on HTTP errors instead of returning bad data
    tracks = response.json()['recenttracks']['track']
    if isinstance(tracks, dict):  # the API returns a bare dict when a page has only one track
        tracks = [tracks]
    rows = [parse_track(track) for track in tracks]
    return [row for row in rows if row is not None]  # drop the now-playing track

In [8]:
test_rows = fetch_page(1)
print(len(test_rows))
test_rows[0]

200


{'artist': 'Radiohead',
 'album': 'In Rainbows',
 'track': 'Jigsaw Falling Into Place',
 'uts': 1794199036}

## Fetch all pages
The number of pages comes from the API response. Set `TEST_PAGES` to a small number for a test run, or leave it as `None` to fetch everything. Failed requests are retried, with a short pause between requests to stay within the API's rate limits.

In [9]:
TEST_PAGES = None  # number of pages for a test run (e.g. 3); None pulls all pages

total_pages = int(data['recenttracks']['@attr']['totalPages'])
pages_to_fetch = range(1, (TEST_PAGES or total_pages) + 1)

all_rows = []
for page in pages_to_fetch:
    for attempt in range(3):  # retry a failed page up to 3 times
        try:
            all_rows.extend(fetch_page(page))
            break  # success, move on to the next page
        except (requests.RequestException, KeyError) as e:  # KeyError: API error body with no 'recenttracks'
            print(f'page {page}, attempt {attempt + 1} failed: {e}')
            time.sleep(2)
    else:  # runs only if all 3 attempts failed
        raise RuntimeError(f'page {page} failed 3 times; stopping')
    time.sleep(0.25)  # be polite to the API

print(f'{len(all_rows)} rows from {len(pages_to_fetch)} pages')

33722 rows from 169 pages


## Build the DataFrame
One row per scrobble. The Unix timestamp (`uts`, UTC) is converted to a local-time `played_at` column. Set `TIMEZONE` in `lastfm.env` to your own time zone (UTC if unset).

In [10]:
track_df = pd.DataFrame(all_rows)
print(track_df.shape)

(33722, 4)


In [ ]:
TIMEZONE = os.environ.get('TIMEZONE', 'UTC')  # IANA time zone name, e.g. 'America/New_York'

track_df['played_at'] = pd.to_datetime(track_df['uts'], unit='s', utc=True).dt.tz_convert(TIMEZONE)

## Clean and save
Drop duplicate rows (pages can overlap if new scrobbles come in during the download), sort oldest to newest, and save to Parquet. The file is only written after a full pull.

In [12]:
print('duplicates:', track_df.duplicated().sum())
track_df = (track_df
            .drop_duplicates()
            .sort_values('played_at')
            .reset_index(drop=True))
print(track_df.shape)
if len(pages_to_fetch) == total_pages:  # only overwrite the saved file after a full pull
    track_df.to_parquet('scrobbles.parquet', index=False)
else:
    print('test run, not saving')

duplicates: 0
(33722, 5)


In [13]:
track_df.head()

,artist,album,track,uts,played_at
0,Public Enemy,It Takes a Nation of Millions to Hold Us Back,She Watch Channel Zero?!,1740504880,2025-02-25 10:34:40-07:00
1,Creedence Clearwater Revival,Cosmo's Factory,Run Through the Jungle,1740505697,2025-02-25 10:48:17-07:00
2,Madvillain,Madvillainy,America's Most Blunted (feat. Quasimoto),1740505945,2025-02-25 10:52:25-07:00
3,The Decemberists,The Crane Wife (10th Anniversary Edition),Shankill Butchers,1740506232,2025-02-25 10:57:12-07:00
4,Aaliyah,I Care 4 U,Are You That Somebody,1740507492,2025-02-25 11:18:12-07:00


## Artist metadata
Top tags and global stats for each artist. Raw API responses are cached as JSON lines, so an interrupted pull can resume and the parsing can be re-run without new API calls.

## Artist API helpers
`call_api` retries temporary errors such as rate limits. `fetch_artist` gets an artist's tags and stats, without the bio text.

In [14]:
CACHE_FILE = 'cache/artist_metadata.jsonl'
os.makedirs('cache', exist_ok=True)

RETRYABLE_ERRORS = {8, 11, 16, 29}  # temporary API errors: operation failed, service offline, try again, rate limit


def call_api(method, **extra_params):
    '''Call a Last.fm API method and return the parsed JSON, retrying temporary failures.'''
    request_params = {'method': method, 'api_key': API_KEY, 'format': 'json', **extra_params}
    for attempt in range(3):
        try:
            result = requests.get(URL, params=request_params, timeout=30).json()
            if result.get('error') in RETRYABLE_ERRORS:
                raise RuntimeError(result.get('message', 'temporary API error'))
            time.sleep(0.25)  # stay well under the API rate limit
            return result  # permanent errors (e.g. artist not found) are returned as-is
        except (requests.RequestException, ValueError, RuntimeError) as e:
            print(f'{method} attempt {attempt + 1} failed: {e}')
            time.sleep(2 * (attempt + 1))
    raise RuntimeError(f'{method} failed 3 times')


def fetch_artist(name):
    '''Fetch top tags and global stats for one artist.'''
    top_tags = call_api('artist.getTopTags', artist=name, autocorrect=1)
    info = call_api('artist.getInfo', artist=name, autocorrect=1)
    if 'artist' in info:
        info['artist'].pop('bio', None)  # long text that isn't used
    return {'artist': name, 'top_tags': top_tags, 'info': info}

In [15]:
sample = fetch_artist('Talking Heads')
print(sample['top_tags']['toptags']['tag'][:3])
print(sample['info']['artist']['stats'])

[{'count': 100, 'name': 'new wave', 'url': 'https://www.last.fm/tag/new+wave'}, {'count': 79, 'name': 'post-punk', 'url': 'https://www.last.fm/tag/post-punk'}, {'count': 26, 'name': '80s', 'url': 'https://www.last.fm/tag/80s'}]
{'listeners': '3359033', 'playcount': '149631721'}


## Fetch all artists
Artists already in the cache are skipped, so the cell can be re-run after an interruption. Set `TEST_ARTISTS` to a small number for a test run, or leave it as `None` to fetch every artist that isn't cached yet. Expect roughly a second or two per artist.

In [16]:
TEST_ARTISTS = None  # number of artists for a test run (e.g. 5); None fetches every uncached artist

artists = sorted(pd.read_parquet('scrobbles.parquet', columns=['artist'])['artist'].unique())

done = set()
if os.path.exists(CACHE_FILE):
    with open(CACHE_FILE) as f:
        done = {json.loads(line)['artist'] for line in f}

todo = [name for name in artists if name not in done]
print(f'{len(artists)} artists, {len(done)} cached, {len(todo)} to fetch')

artists_to_fetch = todo[:TEST_ARTISTS]  # None slices to the whole list

with open(CACHE_FILE, 'a') as f:
    for i, name in enumerate(artists_to_fetch, start=1):
        f.write(json.dumps(fetch_artist(name)) + '\n')
        f.flush()  # write to disk immediately so an interruption loses nothing
        if i % 100 == 0:
            print(f'{i}/{len(artists_to_fetch)} fetched')

1737 artists, 314 cached, 1423 to fetch
100/1423 fetched
200/1423 fetched
300/1423 fetched
400/1423 fetched
500/1423 fetched
600/1423 fetched
700/1423 fetched
800/1423 fetched
900/1423 fetched
1000/1423 fetched
1100/1423 fetched
1200/1423 fetched
1300/1423 fetched
1400/1423 fetched


## Parse and save
Tags go in a long table (one row per artist and tag), and artist stats go in their own table. Both are only written after every artist has been fetched.

In [17]:
def parse_tags(record):
    '''One row per (artist, tag), with the tag weight (0-100).'''
    tags = record['top_tags'].get('toptags', {}).get('tag', [])
    if isinstance(tags, dict):  # a single tag can come back as a bare dict
        tags = [tags]
    return [{'artist': record['artist'], 'tag': t['name'].lower(), 'weight': int(t['count'])} for t in tags]


def parse_stats(record):
    '''Global listener and play counts for one artist (None values if the artist wasn't found).'''
    info = record['info'].get('artist')
    if info is None:
        return {'artist': record['artist'], 'artist_mbid': None, 'listeners': None, 'global_plays': None}
    return {
        'artist': record['artist'],
        'artist_mbid': info.get('mbid') or None,
        'listeners': int(info['stats']['listeners']),
        'global_plays': int(info['stats']['playcount']),
    }

In [18]:
with open(CACHE_FILE) as f:
    records = [json.loads(line) for line in f]

tags_df = pd.DataFrame([row for record in records for row in parse_tags(record)])
artists_df = pd.DataFrame([parse_stats(record) for record in records])
print(tags_df.shape, artists_df.shape)

if len(records) == len(artists):  # only save once every artist is cached
    tags_df.to_parquet('artist_tags.parquet', index=False)
    artists_df.to_parquet('artists.parquet', index=False)
else:
    print(f'{len(records)} of {len(artists)} artists cached, not saving')
tags_df.head()

(13724, 3) (1737, 4)


,artist,tag,weight
0,"10,000 Maniacs",female vocalists,100
1,"10,000 Maniacs",alternative,93
2,"10,000 Maniacs",rock,40
3,"10,000 Maniacs",pop,37
4,"10,000 Maniacs",80s,17


In [19]:
artists_df.head()

,artist,artist_mbid,listeners,global_plays
0,"10,000 Maniacs",b9a06530-1241-4162-836f-7b8e79deaa58,538602,6525593
1,2Pac,382f1005-e9ab-4684-afd4-0bdae4ee37f2,4330228,153576951
2,A$AP Rocky,25b7b584-d952-4662-a8b9-dd8cdfbfeb64,3639360,396995078
3,Abhi The Nomad,NaN,253316,3915167
4,Abhi The Nomad & Kato On The Track,NaN,94,1540


In [20]:
for name in ['scrobbles', 'artists', 'artist_tags']:
    print(name, pd.read_parquet(f'{name}.parquet').shape)

scrobbles (33722, 5)
artists (1737, 4)
artist_tags (13724, 3)


## Sample data
A slice of the most recent scrobbles, with the matching artist and tag rows, so the other notebooks can run without an API key. The slice is contiguous rather than random, so patterns by time of day stay intact.

In [27]:
SAMPLE_DAYS = 120  # most recent days of scrobbles to include in the shared sample

full = pd.read_parquet('scrobbles.parquet')
cutoff = full['played_at'].max() - pd.Timedelta(days=SAMPLE_DAYS)
sample = full[full['played_at'] > cutoff]

sample_artists = set(sample['artist'])
artists = pd.read_parquet('artists.parquet')
tags = pd.read_parquet('artist_tags.parquet')

os.makedirs('sample_data', exist_ok=True)
sample.to_parquet('sample_data/scrobbles.parquet', index=False)
artists[artists['artist'].isin(sample_artists)].to_parquet('sample_data/artists.parquet', index=False)
tags[tags['artist'].isin(sample_artists)].to_parquet('sample_data/artist_tags.parquet', index=False)
print(f'{len(sample):,} scrobbles, {len(sample_artists):,} artists')

3,718 scrobbles, 844 artists
